# Healthcare Data Analysis — Hypothetical Case Study

**Purpose:** Demonstrate how healthcare data can be analysed and translated into practical recommendations for hospital operations and policy.

> **Important:** All records and results in this notebook are synthetic and created for demonstration. They are not real patient data, medical evidence, or findings about a real hospital.

## Project questions
1. What do emergency department (ED) waiting times and throughput look like?
2. How do inpatient length of stay and 30-day readmissions vary?
3. Is completed follow-up within seven days associated with readmission in this synthetic dataset?
4. How can results be communicated responsibly to non-technical stakeholders?

## Workflow
1. Create reproducible synthetic data.
2. Inspect and validate data quality.
3. Calculate descriptive metrics.
4. Explore differences and associations.
5. Visualise the results.
6. Summarise actionable recommendations and limitations.

## 1. Import libraries

The notebook uses common Python data-analysis libraries: `pandas` for tabular data, `numpy` for numerical operations, and `matplotlib` for charts. If a library is missing in your environment, install it with `pip install pandas numpy matplotlib`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
rng = np.random.default_rng(SEED)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("Libraries imported successfully.")

## 2. Generate a synthetic dataset

We simulate 5,000 ED encounters and 2,000 inpatient discharges over a 12-month period. The values are intentionally illustrative. Because the data are simulated, any patterns found here are built from assumptions and should be treated as a learning exercise—not as real-world evidence.

In [ ]:
# Synthetic ED encounter data
n_ed = 5000
start_date = pd.Timestamp("2025-01-01")
ed_dates = start_date + pd.to_timedelta(rng.integers(0, 365, n_ed), unit="D")
hours = rng.choice(np.arange(24), size=n_ed, p=np.array(
    [2,1,1,1,1,2,3,4,5,5,5,5,5,5,5,5,6,7,7,6,5,4,3,2], dtype=float
) / np.sum([2,1,1,1,1,2,3,4,5,5,5,5,5,5,5,5,6,7,7,6,5,4,3,2]))

triage = rng.choice(["Low", "Medium", "High"], size=n_ed, p=[0.35, 0.48, 0.17])
triage_wait_effect = pd.Series(triage).map({"Low": 18, "Medium": 5, "High": -12}).to_numpy()
peak_effect = np.where((hours >= 16) & (hours <= 21), 15, 0)
wait_minutes = np.maximum(5, rng.normal(48, 22, n_ed) + peak_effect + triage_wait_effect)
ed_length_minutes = np.maximum(wait_minutes + 20, rng.normal(220, 65, n_ed) + peak_effect)
ed = pd.DataFrame({
    "encounter_id": [f"ED{i:05d}" for i in range(1, n_ed + 1)],
    "arrival_date": pd.to_datetime(ed_dates),
    "arrival_hour": hours,
    "triage_category": triage,
    "wait_minutes": np.round(wait_minutes, 1),
    "total_ed_minutes": np.round(ed_length_minutes, 1),
    "completed_within_4_hours": ed_length_minutes <= 240,
    "left_without_being_seen": rng.random(n_ed) < (0.025 + np.where(wait_minutes > 75, 0.035, 0)),
})
ed["month"] = ed["arrival_date"].dt.to_period("M").astype(str)

# Synthetic inpatient discharge data
n_ip = 2000
ip_dates = start_date + pd.to_timedelta(rng.integers(0, 365, n_ip), unit="D")
age_band = rng.choice(["18–44", "45–64", "65+"], size=n_ip, p=[0.30, 0.34, 0.36])
risk = rng.choice(["Lower", "Higher"], size=n_ip, p=[0.68, 0.32])
followup_prob = np.where(risk == "Higher", 0.52, 0.62)
followup_7d = rng.random(n_ip) < followup_prob
length_stay = np.maximum(1, rng.gamma(shape=2.1, scale=2.0, size=n_ip) + np.where(risk == "Higher", 1.3, 0))
# Illustrative probability structure; not a clinical model
readmit_prob = (
    0.095
    + np.where(risk == "Higher", 0.095, 0)
    + np.where(followup_7d, -0.025, 0.025)
    + np.where(age_band == "65+", 0.025, 0)
)
readmit_prob = np.clip(readmit_prob, 0.02, 0.60)
readmitted = rng.random(n_ip) < readmit_prob

inpatient = pd.DataFrame({
    "discharge_id": [f"IP{i:05d}" for i in range(1, n_ip + 1)],
    "discharge_date": pd.to_datetime(ip_dates),
    "age_band": age_band,
    "risk_group": risk,
    "length_of_stay_days": np.round(length_stay, 1),
    "followup_completed_within_7d": followup_7d,
    "unplanned_readmission_within_30d": readmitted,
    "patient_experience_positive": rng.random(n_ip) < np.where(risk == "Higher", 0.75, 0.82),
})
inpatient["month"] = inpatient["discharge_date"].dt.to_period("M").astype(str)

print(f"ED encounters: {len(ed):,}")
print(f"Inpatient discharges: {len(inpatient):,}")
display(ed.head())
display(inpatient.head())

## 3. Data quality checks

Check dataset dimensions, duplicate identifiers, missing values, and basic ranges before interpreting results. In a real project, these checks should be documented and reconciled against source-system totals.

In [ ]:
def quality_summary(df, id_col, numeric_cols):
    return {
        "rows": len(df),
        "duplicate_ids": int(df[id_col].duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "numeric_minima": {col: float(df[col].min()) for col in numeric_cols},
        "numeric_maxima": {col: float(df[col].max()) for col in numeric_cols},
    }

print("ED quality summary:")
display(quality_summary(ed, "encounter_id", ["wait_minutes", "total_ed_minutes"]))
print("\nInpatient quality summary:")
display(quality_summary(inpatient, "discharge_id", ["length_of_stay_days"]))

assert ed["encounter_id"].is_unique
assert inpatient["discharge_id"].is_unique
assert ed["wait_minutes"].ge(0).all()
assert inpatient["length_of_stay_days"].gt(0).all()
print("\nBasic validation checks passed.")

## 4. Baseline metrics

We calculate medians and percentages alongside denominators. A metric should always be interpreted with its definition, eligible population, time window, and data-quality caveats.

In [ ]:
ed_metrics = pd.DataFrame({
    "Metric": [
        "Median ED arrival-to-clinician wait (minutes)",
        "ED encounters completed within 4 hours",
        "ED encounters where patient left before being seen",
    ],
    "Value": [
        ed["wait_minutes"].median(),
        ed["completed_within_4_hours"].mean() * 100,
        ed["left_without_being_seen"].mean() * 100,
    ],
    "Unit": ["minutes", "%", "%"],
    "Denominator": [len(ed), len(ed), len(ed)],
})

ip_metrics = pd.DataFrame({
    "Metric": [
        "Mean inpatient length of stay (days)",
        "30-day unplanned readmission",
        "Follow-up completed within 7 days",
        "Positive patient-experience response",
    ],
    "Value": [
        inpatient["length_of_stay_days"].mean(),
        inpatient["unplanned_readmission_within_30d"].mean() * 100,
        inpatient["followup_completed_within_7d"].mean() * 100,
        inpatient["patient_experience_positive"].mean() * 100,
    ],
    "Unit": ["days", "%", "%", "%"],
    "Denominator": [len(inpatient)] * 4,
})

print("Emergency department metrics")
display(ed_metrics.round(2))
print("Inpatient and transition-of-care metrics")
display(ip_metrics.round(2))

## 5. Visualise ED access and throughput

The charts show distributions and monthly patterns. A median alone can hide very long waits, so distribution and percentile measures are useful complements.

In [ ]:
# Distribution of ED waiting times
plt.figure(figsize=(9, 5))
plt.hist(ed["wait_minutes"], bins=35)
plt.title("Synthetic ED Arrival-to-Clinician Waiting Time")
plt.xlabel("Waiting time (minutes)")
plt.ylabel("Number of encounters")
plt.tight_layout()
plt.show()

# Monthly ED indicators
monthly_ed = ed.groupby("month").agg(
    median_wait_minutes=("wait_minutes", "median"),
    completed_within_4h_pct=("completed_within_4_hours", lambda x: x.mean() * 100),
    encounter_count=("encounter_id", "count"),
).reset_index()

fig = plt.figure(figsize=(11, 5))
plt.plot(monthly_ed["month"], monthly_ed["median_wait_minutes"], marker="o")
plt.title("Monthly Median ED Waiting Time")
plt.xlabel("Month")
plt.ylabel("Median wait (minutes)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

display(monthly_ed.round(2))

## 6. Explore ED variation by arrival hour and triage category

This descriptive comparison can help formulate operational questions. It does not establish that staffing or time of day caused a delay.

In [ ]:
hourly_ed = ed.groupby("arrival_hour").agg(
    encounters=("encounter_id", "count"),
    median_wait=("wait_minutes", "median"),
    completed_within_4h_pct=("completed_within_4_hours", lambda x: x.mean() * 100),
).reset_index()

fig = plt.figure(figsize=(10, 5))
plt.bar(hourly_ed["arrival_hour"], hourly_ed["encounters"])
plt.title("Synthetic ED Arrival Volume by Hour")
plt.xlabel("Hour of day")
plt.ylabel("Number of encounters")
plt.xticks(range(24))
plt.tight_layout()
plt.show()

triage_summary = ed.groupby("triage_category").agg(
    encounters=("encounter_id", "count"),
    median_wait_minutes=("wait_minutes", "median"),
    completed_within_4h_pct=("completed_within_4_hours", lambda x: x.mean() * 100),
).reset_index()
display(triage_summary.round(2))

## 7. Inpatient length of stay and readmissions

Length of stay and readmission rates can vary with patient mix and clinical complexity. These simple group comparisons are unadjusted and should not be used to rank clinical teams.

In [ ]:
fig = plt.figure(figsize=(8, 5))
plt.hist(inpatient["length_of_stay_days"], bins=30)
plt.title("Synthetic Inpatient Length of Stay")
plt.xlabel("Length of stay (days)")
plt.ylabel("Number of discharges")
plt.tight_layout()
plt.show()

readmit_by_risk = inpatient.groupby("risk_group").agg(
    discharges=("discharge_id", "count"),
    readmission_rate_pct=("unplanned_readmission_within_30d", lambda x: x.mean() * 100),
    followup_7d_pct=("followup_completed_within_7d", lambda x: x.mean() * 100),
    mean_los_days=("length_of_stay_days", "mean"),
).reset_index()

display(readmit_by_risk.round(2))

## 8. Follow-up and readmission: association, not causation

Compare readmission rates among people with and without completed follow-up. Because the synthetic data-generation process intentionally makes follow-up status part of the readmission probability, an association is expected by design. In real data, confounding, access barriers, illness severity, and incomplete outcome capture would need careful consideration.

In [ ]:
followup_summary = inpatient.groupby("followup_completed_within_7d").agg(
    discharges=("discharge_id", "count"),
    readmission_rate_pct=("unplanned_readmission_within_30d", lambda x: x.mean() * 100),
).reset_index()

followup_summary["followup_status"] = followup_summary["followup_completed_within_7d"].map({
    True: "Follow-up completed within 7 days",
    False: "No completed follow-up within 7 days",
})
display(followup_summary[["followup_status", "discharges", "readmission_rate_pct"]].round(2))

plt.figure(figsize=(8, 5))
plt.bar(followup_summary["followup_status"], followup_summary["readmission_rate_pct"])
plt.title("Readmission Rate by 7-Day Follow-up Status (Synthetic Data)")
plt.xlabel("")
plt.ylabel("30-day readmission rate (%)")
plt.xticks(rotation=12, ha="right")
plt.tight_layout()
plt.show()

## 9. Patient experience and subgroup monitoring

Subgroup comparisons can reveal where further investigation may be needed. Before acting, assess sample sizes, response rates, measurement consistency, and privacy risk. Do not interpret demographic or group differences as evidence of patient blame.

In [ ]:
experience_summary = inpatient.groupby("age_band").agg(
    discharges=("discharge_id", "count"),
    positive_experience_pct=("patient_experience_positive", lambda x: x.mean() * 100),
    readmission_rate_pct=("unplanned_readmission_within_30d", lambda x: x.mean() * 100),
).reset_index()

display(experience_summary.round(2))

plt.figure(figsize=(7, 5))
plt.bar(experience_summary["age_band"], experience_summary["positive_experience_pct"])
plt.title("Positive Patient Experience by Age Band (Synthetic Data)")
plt.xlabel("Age band")
plt.ylabel("Positive responses (%)")
plt.tight_layout()
plt.show()

## 10. Automatically generated summary

The following summary uses the simulated data generated above. Its numbers may change if you modify the random seed or data-generation assumptions.

In [ ]:
summary = {
    "ED encounters analysed": len(ed),
    "Median ED wait (minutes)": round(float(ed["wait_minutes"].median()), 1),
    "ED completed within 4 hours (%)": round(float(ed["completed_within_4_hours"].mean() * 100), 1),
    "Left without being seen (%)": round(float(ed["left_without_being_seen"].mean() * 100), 1),
    "Inpatient discharges analysed": len(inpatient),
    "Mean length of stay (days)": round(float(inpatient["length_of_stay_days"].mean()), 2),
    "30-day readmission (%)": round(float(inpatient["unplanned_readmission_within_30d"].mean() * 100), 1),
    "Follow-up within 7 days (%)": round(float(inpatient["followup_completed_within_7d"].mean() * 100), 1),
    "Positive experience (%)": round(float(inpatient["patient_experience_positive"].mean() * 100), 1),
}
for metric, value in summary.items():
    print(f"{metric}: {value}")

## 11. Recommendations linked to the analysis

Use these as hypotheses for a real project, not as automatic conclusions from this simulated notebook.

1. **ED flow:** Inspect hourly arrival patterns, triage mix, diagnostics, boarding, and staffing. Pilot workflow changes only with clinical oversight.
2. **Transitions of care:** Review readmissions and follow-up barriers. Test appropriate follow-up scheduling, medication reconciliation, clear discharge instructions, and post-discharge contact for selected patients.
3. **Inpatient flow:** Review discharge delays by reason and compare length of stay within clinically comparable groups.
4. **Equity monitoring:** Track access, outcomes, follow-up, and experience across relevant patient groups while protecting privacy.
5. **Data quality:** Standardise definitions, validate timestamps and identifiers, document exclusions, and publish denominators with every rate.
6. **Evaluation:** Define outcomes and balancing measures before a pilot. Use run charts and, where feasible, a comparison group or interrupted time-series design.

## 12. Limitations and future work

- All data are synthetic; results are not real-world evidence.
- The simulated relationships are partly imposed by the data-generation process.
- No confidence intervals, formal hypothesis tests, or risk-adjusted clinical models are included in this introductory notebook.
- Readmission capture is assumed complete, unlike many real-world settings.
- Follow-up status and outcomes are simplified and may not reflect clinical appropriateness.
- Real analysis requires approved data access, privacy safeguards, data-quality review, clinical input, and a pre-specified analysis plan.
- Future work could use validated local data, risk adjustment, missing-data analysis, patient interviews, cost analysis, and stronger evaluation designs.

## 13. Export summary tables

The following cell exports summary tables to CSV so they can be added to a report or spreadsheet.

In [ ]:
ed_metrics.to_csv("ed_baseline_metrics.csv", index=False)
ip_metrics.to_csv("inpatient_baseline_metrics.csv", index=False)
monthly_ed.to_csv("monthly_ed_metrics.csv", index=False)
readmit_by_risk.to_csv("readmission_by_risk_group.csv", index=False)
print("CSV summary files exported to the current working directory.")

## Final takeaway

A useful healthcare analytics workflow moves from **validated measures → careful interpretation → a testable improvement hypothesis → monitored pilot → evaluation**. Dashboard metrics should support clinical judgement and patient-centred decisions, not replace them.